# ML EDA Full Fledge — Complete Setup, Training, Evaluation & Flask API

This notebook gives the complete sequence from a fresh VS Code/PowerShell session through:
- activating the `.venv` environment
- checking the project structure
- installing/checking dependencies
- training Logistic Regression, Random Forest and XGBoost
- generating Accuracy, Precision, Recall, F1 and ROC-AUC reports
- starting the Flask prediction API
- testing `/predict`

**Project root used in this project:** `D:\\BW\\ML_Project\\ML_EDA_full_fledge`

**Important:** Run the PowerShell cells in a VS Code PowerShell terminal. For Google Colab, use the Colab-specific cells at the end; local Windows paths such as `D:\\...` do not work in Colab.

## 1. Fresh VS Code window — open the project folder


In [1]:
# PowerShell command
cd "D:\\BW\\ML_Project\\ML_EDA_full_fledge"
pwd


SyntaxError: invalid syntax (1596779322.py, line 2)

### Expected location


In [ ]:
# PowerShell
Get-Location
dir


## 2. Activate the virtual environment

`deactivate` is the correct command. `deactive` is a typo.


In [ ]:
# PowerShell
& ".\\.venv\\Scripts\\Activate.ps1"


If PowerShell blocks activation, run this once for the current terminal and then activate again.

In [ ]:
# PowerShell, only if activation is blocked
Set-ExecutionPolicy -Scope Process -ExecutionPolicy RemoteSigned
& ".\\.venv\\Scripts\\Activate.ps1"


## 3. Verify Python and the selected environment


In [ ]:
# PowerShell
python --version
python -c "import sys; print(sys.executable)"
python -m pip --version


Expected Python executable should point inside:

`D:\\BW\\ML_Project\\ML_EDA_full_fledge\\.venv\\Scripts\\python.exe`

## 4. Select the same interpreter in VS Code

1. Press `Ctrl + Shift + P`.
2. Select **Python: Select Interpreter**.
3. Choose the project `.venv` interpreter.
4. The path should end with `.venv\\Scripts\\python.exe`.


## 5. Check/install required packages

Run the checks first. If a package is missing, install it in the active `.venv`.

In [ ]:
# PowerShell
python -c "import pandas, numpy, sklearn, joblib, flask, xgboost; print('All required packages are OK')"
python -c "import xgboost; print('XGBoost:', xgboost.__version__)"


In [ ]:
# PowerShell — use only if imports above fail
python -m pip install pandas numpy scikit-learn joblib flask xgboost


## 6. Check the important project files


In [ ]:
# PowerShell
dir src
dir models
dir reports
dir logs


Your `src` folder should contain files such as `app.py`, `config.py`, `features.py`, `dataset.py`, `logging_config.py`, and `train_models.py`.

The training code should save models under the version configured by `VERSION_MODEL_DIR`. Your recent run showed `models\\v1.0.0`, so use the configured path rather than manually assuming `models\\v1`.

## 7. Train all three ML algorithms

This step creates/updates:
- Logistic Regression model
- Random Forest model
- XGBoost model
- fitted preprocessor
- evaluation reports


In [ ]:
# PowerShell — run from project root
python -m src.train_models


Wait until the terminal shows **All models trained successfully.**

## 8. Check generated model files


In [ ]:
# PowerShell
Get-ChildItem models -Recurse -File | Select-Object FullName, Length


You should have files similar to:

`logistic_regression.joblib`  
`random_forest.joblib`  
`xgboost.joblib`  
`preprocessor.joblib`

The exact version folder comes from `VERSION_MODEL_DIR`.

## 9. Check generated reports


In [ ]:
# PowerShell
Get-ChildItem reports -Recurse -File | Select-Object FullName, Length


## 10. Read Accuracy, Precision, Recall, F1 and ROC-AUC

If your reports are in `reports\\v1`, use the commands below. If your project uses another report version, replace `v1` with that folder name.

In [ ]:
# PowerShell — example for reports/v1
Get-Content reports\\v1\\logistic_regression_report.json
Get-Content reports\\v1\\random_foGet-Content reports\\v1\\random_forest_report.json
rest_report.json
Get-Content reports\\v1\\xgboost_report.json


### Metrics meaning

- **Accuracy:** percentage of all predictions that are correct.
- **Precision:** among predicted defaults, how many were actually defaults.
- **Recall:** among actual defaults, how many were detected.
- **F1 Score:** balance between precision and recall.
- **ROC-AUC:** how well the model separates the two classes across thresholds.


## 11. Compare all three models

If `src/evaluate_models.py` exists, run it.

In [ ]:
# PowerShell
python -m src.evaluate_models


If that file does not exist, create it with the following code.

In [ ]:
# Save this as src/evaluate_models.py
import json
from pathlib import Path
import pandas as pd

REPORT_ROOT = Path('reports')
models = ['logistic_regression', 'random_forest', 'xgboost']
    
for version_dir in sorted(REPORT_ROOT.iterdir()):
    if not version_dir.is_dir():
        continue
    rows = []
    for model_name in models:
        path = version_dir / f'{model_name}_report.json'
        if path.exists():
            with open(path, encoding='utf-8') as f:
                r = json.load(f)
            rows.append({
                'Model': model_name,
                'Accuracy': r.get('accuracy'),
                'Precision': r.get('precision'),
                'Recall': r.get('recall'),
                'F1 Score': r.get('f1_score'),
                'ROC-AUC': r.get('roc_auc'),
            })
    if rows:
        df = pd.DataFrame(rows)
        print(f'\n=== MODEL VERSION: {version_dir.name} ===')
        print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
        best = df.loc[df['F1 Score'].idxmax()]
        print(f"\nBest F1 model: {best['Model']} ({best['F1 Score']:.4f})")


## 12. Start the Flask prediction API

Because `app.py` imports modules using `from src...`, start it as a module from the project root.

In [ ]:
# PowerShell — Terminal 1
python -m src.app


Keep Terminal 1 running. You should see Flask listening on `http://127.0.0.1:5000`.

## 13. Open a second terminal and activate `.venv`


In [ ]:
# PowerShell — Terminal 2
cd "D:\\BW\\ML_Project\\ML_EDA_full_fledge"
& ".\\.venv\\Scripts\\Activate.ps1"


## 14. Test the Flask API

Your `test_api.py` should POST a JSON record to `/predict`.

In [ ]:
# PowerShell — Terminal 2
python test_api.py


Expected successful response:

`Status Code: 200`

and predictions/probabilities for Logistic Regression, Random Forest and XGBoost.

If you see the old **feature names should match** error, make sure `app.py` loads `preprocessor.joblib` from the same model version used to train the three models and calls `PREPROCESSOR.transform(df_model)` before prediction.

## 15. Check API health without `test_api.py`


In [ ]:
# PowerShell — with Flask running
Invoke-RestMethod http://127.0.0.1:5000/health


## 16. Check Git status

After all code/notebook/report changes are complete:

In [ ]:
# PowerShell
git status
git add src notebooks reports .gitignore
git status
git commit -m "Add model versioning logging Flask API and model reports"
git push


## 17. Deactivate environment when finished


In [ ]:
# PowerShell
deactivate


# Google Colab notes

For Google Colab, upload your project/repository or clone your GitHub repository first. Do **not** use the Windows `D:\\BW\\...` path. Colab uses Linux paths such as `/content/ML_EDA_full_fledge`.

In [ ]:
# Colab cell 1 — clone your GitHub repository
!git clone https://github.com/sanika-kadam10/ML_EDA_full_fledge.git
%cd /content/ML_EDA_full_fledge


In [ ]:
# Colab cell 2 — install project dependencies
!pip install -q pandas numpy scikit-learn joblib flask xgboost


In [ ]:
# Colab cell 3 — verify packages
import sys, pandas, numpy, sklearn, joblib, xgboost
print('Python:', sys.version)
print('pandas:', pandas.__version__)
print('numpy:', numpy.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__)
print('xgboost:', xgboost.__version__)


In [ ]:
# Colab cell 4 — inspect project
!find . -maxdepth 3 -type f | head -100


In [ ]:
# Colab cell 5 — train models
!python -m src.train_models


In [ ]:
# Colab cell 6 — inspect generated models/reports
!find models reports -type f | sort


### Important Colab API note

A Flask server started in Colab is not directly reachable from your local PC at `127.0.0.1:5000`. For Colab API testing, you need a tunnel such as ngrok or test the prediction function directly inside Colab. Your normal local Windows workflow should use `python -m src.app` + `python test_api.py`.

# Final workflow

**Every fresh VS Code session:**

1. Open `D:\\BW\\ML_Project\\ML_EDA_full_fledge`.
2. Activate `.venv`.
3. Verify Python interpreter.
4. Check dependencies.
5. Run `python -m src.train_models` when retraining is needed.
6. Check model/report files.
7. Start API with `python -m src.app`.
8. In a second terminal run `python test_api.py`.
9. Review Accuracy/F1/ROC-AUC reports.
10. Commit and push changes.
11. Run `deactivate` when finished.
